# FINANCE DATABASE GENERATION

Name : Ibrahim Abdulsalam


In [ ]:
# IMPORT LIBRARIES
!pip install faker
import sqlite3
import random
from faker import Faker
from datetime import datetime

fake = Faker()

conn = sqlite3.connect("finance.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.6 MB/s eta 0:00:00


In [ ]:
# CREATE TABLES
cursor.executescript("""

-- Create Customers table to store individual customer details
CREATE TABLE IF NOT EXISTS Customers (
    customer_id INTEGER PRIMARY KEY, -- Unique identifier for each customer
    first_name TEXT NOT NULL, -- Customer's first name, cannot be empty
    last_name TEXT NOT NULL, -- Customer's last name, cannot be empty
    email TEXT UNIQUE, -- Customer's email address, must be unique across customers (can be NULL if not provided)
    gender TEXT CHECK(gender IN ('Male','Female','Other')), -- Customer's gender, restricted to specific values
    date_of_birth DATE -- Customer's date of birth
);

-- Create Accounts table to store bank account details
CREATE TABLE IF NOT EXISTS Accounts (
    account_id INTEGER PRIMARY KEY, -- Unique identifier for each account
    account_type TEXT CHECK(account_type IN ('Checking','Savings','Investment')), -- Type of account, restricted to specific values
    balance REAL CHECK(balance >= 0), -- Current balance of the account, must be non-negative
    currency TEXT CHECK(currency IN ('USD','EUR','GBP')), -- Currency of the account, restricted to specific values
    account_status TEXT CHECK(account_status IN ('Active','Suspended','Closed')) -- Current status of the account, restricted to specific values
);

-- MANY-TO-MANY LINK TABLE
-- Create AccountHolders table to link customers to accounts (many-to-many relationship)
-- This table supports joint accounts where multiple customers can hold the same account, or a single customer can have multiple accounts.
CREATE TABLE IF NOT EXISTS AccountHolders (
    customer_id INTEGER, -- Foreign key referencing Customers table
    account_id INTEGER, -- Foreign key referencing Accounts table
    holder_role TEXT CHECK(holder_role IN ('Primary','Secondary')), -- Role of the customer in relation to the account
    PRIMARY KEY (customer_id, account_id), -- Composite primary key to ensure unique customer-account pairing
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id), -- Enforce referential integrity with Customers table
    FOREIGN KEY (account_id) REFERENCES Accounts(account_id) -- Enforce referential integrity with Accounts table
);

-- Create TransactionCategories table to categorize financial transactions
CREATE TABLE IF NOT EXISTS TransactionCategories (
    category_id INTEGER PRIMARY KEY, -- Unique identifier for each transaction category
    category_name TEXT, -- Name of the transaction category (e.g., 'Salary', 'Rent')
    category_type TEXT CHECK(category_type IN ('Income','Expense','Transfer')) -- Type of the category, restricted to specific values
);

-- Create Transactions table to record all financial transactions
CREATE TABLE IF NOT EXISTS Transactions (
    transaction_id INTEGER, -- Unique identifier for the transaction (part of composite primary key)
    account_id INTEGER, -- Foreign key referencing Accounts table (part of composite primary key)
    category_id INTEGER, -- Foreign key referencing TransactionCategories table
    amount REAL CHECK(amount >= 0), -- Amount of the transaction, must be non-negative
    transaction_date DATE, -- Date when the transaction occurred
    description TEXT, -- Description of the transaction
    payment_method TEXT CHECK(payment_method IN ('Cash','Card','Transfer')), -- Method used for the payment, restricted to specific values
    PRIMARY KEY (transaction_id, account_id), -- Composite primary key to uniquely identify each transaction within an account
    FOREIGN KEY (account_id) REFERENCES Accounts(account_id), -- Enforce referential integrity with Accounts table
    FOREIGN KEY (category_id) REFERENCES TransactionCategories(category_id) -- Enforce referential integrity with TransactionCategories table
);

-- Create Investments table to track customer investments
CREATE TABLE IF NOT EXISTS Investments (
    investment_id INTEGER PRIMARY KEY, -- Unique identifier for each investment
    customer_id INTEGER, -- Foreign key referencing Customers table
    investment_type TEXT CHECK(investment_type IN ('Stock','Bond','Mutual Fund')), -- Type of investment, restricted to specific values
    amount REAL CHECK(amount >= 0), -- Amount invested, must be non-negative
    purchase_date DATE, -- Date when the investment was made
    risk_level TEXT CHECK(risk_level IN ('Low','Medium','High')), -- Risk level associated with the investment, restricted to specific values
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id) -- Enforce referential integrity with Customers table
);

""")

In [ ]:
# INSERT CUSTOMERS (500)
# Delete all records from the AccountHolders table to prevent foreign key constraint violations on Customers.
cursor.execute("DELETE FROM AccountHolders;")

# Delete all records from the Investments table to prevent foreign key constraint violations on Customers.
cursor.execute("DELETE FROM Investments;")

# Delete all records from the Customers table to ensure a clean slate before inserting new data.
cursor.execute("DELETE FROM Customers;")

# Create a set to store generated emails to ensure uniqueness within this batch
generated_emails_in_batch = set()

for i in range(500):
    # Initialize email for insertion as None by default
    email_for_insertion = None

    # Deliberately set approximately 3% of emails to None to simulate missing data
    if random.random() < 0.03:
        email_for_insertion = None # Explicitly set to None for missing data
    else:
        # Generate emails until a unique one is found within this batch
        # This prevents 'UNIQUE constraint failed' errors during insertion.
        # Add a safeguard against infinite loops in extremely rare cases where unique emails might be exhausted.
        max_attempts_for_unique_email = 50
        attempt = 0
        while True:
            generated_email = fake.email()
            if generated_email not in generated_emails_in_batch:
                email_for_insertion = generated_email
                generated_emails_in_batch.add(email_for_insertion)
                break
            attempt += 1
            if attempt >= max_attempts_for_unique_email:
                # If unable to generate a unique email after many attempts,
                # fall back to making this email None to avoid constraint error.
                email_for_insertion = None
                break

    # Insert the generated fake customer data into the Customers table
    cursor.execute("""
        INSERT INTO Customers (first_name,last_name,email,gender,date_of_birth)
        VALUES (?,?,?,?,?)
    """, (
        fake.first_name(), # Generate a fake first name
        fake.last_name(),  # Generate a fake last name
        email_for_insertion, # Use the generated unique email or potentially None email
        random.choice(['Male','Female','Other']), # Randomly select a gender
        fake.date_of_birth(minimum_age=18, maximum_age=80).isoformat() # Generate a date of birth for an adult and convert to ISO string
    ))

conn.commit() # Commit the transactions to save the inserted data

In [ ]:
# INSERT ACCOUNTS (700)

# Loop 700 times to insert 700 fake account records into the Accounts table
for i in range(700):
    # Insert the generated fake account data into the Accounts table
    cursor.execute("""
        INSERT INTO Accounts (account_type,balance,currency,account_status)
        VALUES (?,?,?,?)
    """, (
        random.choice(['Checking','Savings','Investment']), # Randomly select an account type from predefined options
        round(random.uniform(100,100000),2), # Generate a random balance between 100 and 100,000, rounded to 2 decimal places
        random.choice(['USD','EUR','GBP']), # Randomly select a currency from predefined options
        random.choice(['Active','Suspended','Closed']) # Randomly select an account status from predefined options
    ))

In [ ]:
# LINK ACCOUNT HOLDERS
for account_id in range(1,701):
    primary_customer = random.randint(1,500)

    cursor.execute("""
        INSERT OR IGNORE INTO AccountHolders VALUES (?,?,?)
    """, (primary_customer, account_id, 'Primary'))

    # 20% chance of joint account
    if random.random() < 0.2:
        secondary_customer = random.randint(1,500)
        if secondary_customer != primary_customer:
            cursor.execute("""
                INSERT OR IGNORE INTO AccountHolders VALUES (?,?,?)
            """, (secondary_customer, account_id, 'Secondary'))
conn.commit() # Commit the transactions to save the inserted data

In [ ]:
# TRANSACTION CATEGORIES
# Clear existing data from Transactions to prevent foreign key constraint issues if this cell is re-run with modifications
cursor.execute("DELETE FROM Transactions;")
# Clear existing data from TransactionCategories to prevent duplicate entries or constraint violations
cursor.execute("DELETE FROM TransactionCategories;")

# Define a list of transaction categories with their names and types
categories = [
    ('Salary','Income'),
    ('Rent','Expense'),
    ('Food','Expense'),
    ('Utilities','Expense'),
    ('Transfer','Transfer')
]

# Loop through each predefined category and insert it into the TransactionCategories table
for name, type_ in categories:
    cursor.execute("""
        INSERT INTO TransactionCategories (category_name,category_type)
        VALUES (?,?)
    """,(name,type_)) # Insert the category name and type into the table

# Commit the transactions to save the inserted data
conn.commit()

In [ ]:
cursor.execute("DELETE FROM Transactions;") # Clear existing data from Transactions to prevent duplicate entries or constraint violations

transaction_counter = 1 # Initialize a counter for unique transaction IDs across all accounts

# Loop through each account to generate transactions
for account_id in range(1,701):
    # Generate a random number of transactions (between 5 and 15) for each account
    for _ in range(random.randint(5,15)):
        # Insert the generated fake transaction data into the Transactions table
        cursor.execute("""
            INSERT INTO Transactions VALUES (?,?,?,?,?,?,?)
        """, (
            transaction_counter, # Unique identifier for the transaction
            account_id, # The account ID to which this transaction belongs
            random.randint(1,5), # Randomly select a category ID (from 1 to 5, assuming 5 categories exist)
            round(random.uniform(10,5000),2), # Generate a random transaction amount between 10 and 5000, rounded to 2 decimal places
            fake.date_between(start_date='-2y', end_date='today').isoformat(), # Generate a random transaction date within the last 2 years and convert to ISO string
            fake.sentence(nb_words=5), # Generate a fake sentence as transaction description
            random.choice(['Cash','Card','Transfer']) # Randomly select a payment method
        ))
        transaction_counter += 1 # Increment the transaction counter for the next transaction

conn.commit() # Commit the transactions to save the inserted data

In [ ]:
# INVESTMENTS (400)

cursor.execute("DELETE FROM Investments;") # Clear existing data from Investments to prevent duplicate entries or constraint violations

# Loop to generate and insert 400 fake investment records
for i in range(400):
    # Insert the generated fake investment data into the Investments table
    cursor.execute("""
        INSERT INTO Investments (customer_id,investment_type,amount,purchase_date,risk_level)
        VALUES (?,?,?,?,?)
    """, (
        random.randint(1,500), # Randomly select an existing customer_id
        random.choice(['Stock','Bond','Mutual Fund']), # Randomly select an investment type
        round(random.uniform(500,50000),2), # Generate a random investment amount between 500 and 50,000, rounded to 2 decimal places
        fake.date_between(start_date='-3y', end_date='today').isoformat(), # Generate a random purchase date within the last 3 years and convert to ISO string
        random.choice(['Low','Medium','High']) # Randomly select a risk level for the investment
    ))

conn.commit() # Commit the transactions to save the inserted data

In [ ]:
# DELIBERATE DUPLICATE EMAILS
# Select 5 customer IDs that currently have a non-null email address.
cursor.execute("SELECT customer_id FROM Customers WHERE email IS NOT NULL LIMIT 5")
# Fetch the selected customer IDs that have a non-null email for updating
customer_ids_to_update = cursor.fetchall()

# Iterate through each fetched customer ID to update their email address
for customer_id_tuple in customer_ids_to_update:
    # Extract the customer_id from the tuple
    customer_id = customer_id_tuple[0]
    # Generate a new unique email for each update to avoid integrity errors
    new_email = fake.email()
    # Update the Customers table with the new email for the specific customer_id
    cursor.execute("""
        UPDATE Customers
        SET email = ?
        WHERE customer_id = ?
    """, (new_email, customer_id))

# Commit the transactions to save all changes made to the database
conn.commit()
# Close the database connection, releasing all associated resources
conn.close()

In [11]:
# SAVE THE DATABASE FILE
from google.colab import files
files.download('finance.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>